In [14]:
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np

import glob
test_sample = "7_mono_0.5_decrease/"
print(sorted(glob.glob("{}cellParameters.*.input".format(test_sample))))



['7_mono_0.5_decrease/cellParameters.0.input', '7_mono_0.5_decrease/cellParameters.1.input', '7_mono_0.5_decrease/cellParameters.10.input', '7_mono_0.5_decrease/cellParameters.11.input', '7_mono_0.5_decrease/cellParameters.12.input', '7_mono_0.5_decrease/cellParameters.13.input', '7_mono_0.5_decrease/cellParameters.14.input', '7_mono_0.5_decrease/cellParameters.15.input', '7_mono_0.5_decrease/cellParameters.16.input', '7_mono_0.5_decrease/cellParameters.17.input', '7_mono_0.5_decrease/cellParameters.18.input', '7_mono_0.5_decrease/cellParameters.19.input', '7_mono_0.5_decrease/cellParameters.2.input', '7_mono_0.5_decrease/cellParameters.20.input', '7_mono_0.5_decrease/cellParameters.21.input', '7_mono_0.5_decrease/cellParameters.22.input', '7_mono_0.5_decrease/cellParameters.23.input', '7_mono_0.5_decrease/cellParameters.24.input', '7_mono_0.5_decrease/cellParameters.25.input', '7_mono_0.5_decrease/cellParameters.3.input', '7_mono_0.5_decrease/cellParameters.4.input', '7_mono_0.5_decre

In [ ]:
## create cross section of a periodic tissue
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob

dir = "7_bidisperse_5_4.9_0.5_increase/"
# dir = "7_mono_0.5_decrease/"
df = pd.read_csv("{}stresses.csv".format(dir))
print(df["CellID"])
training_cell = df["CellID"].to_numpy()[0]
ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
final_iter = max(ids)
file = "{}.bulk.txt".format(final_iter)
sample = PeriodicTissue.from_config(dir,file)
min = FIREminimization.periodic_tissue(sample)
min.load_cell_parameters("cellParameters.{}.input".format(final_iter))

for cellID, cell in sample.cells_.items():
    cell.vtk_scalar_ = cell.s0_
    cell.vtk_scalar_ = stress.calculate_max_shear_stress(sample, cellID)
normal = np.array([1,0,0])
center = sample.cells_[training_cell].center_
print(sample.cells_[training_cell].s0_)
makeSampleCrossSection(sample=min._config, center = center, normal = normal, filename = "cross_section.vtk")
single_cell = sample.extract_cell(training_cell)
makeSampleCrossSection(sample=single_cell, center=center, normal = normal, filename="single_cell_cross_section.vtk")



0    331
Name: CellID, dtype: int64
5.0


In [3]:
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob
dir = "8_0_5_1_sample/"
df = pd.read_csv("{}stresses.csv".format(dir))
print(df["CellID"])
# ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
# ids = sorted(ids)
# final_iter = 33
# for iter in range(final_iter+1):
iter = 33
file = "{}.bulk.txt".format(iter)
sample = PeriodicTissue.from_config(dir,file)
min = FIREminimization.periodic_tissue(sample)
min.load_cell_parameters("cellParameters.{}.input".format(iter))
for i, row in df.iterrows():
    cellID = row["CellID"]
    target_stress = row["Target"]
    cell = sample.cells_[cellID]
    # shear = stress.calculate_max_shear_stress(sample, cellID)
    # cell.max_shear_stress_ = shear
    # vtk_scalar = abs(cell.max_shear_stress_ - target_stress)/ target_stress
    for polygonID in cell.polygons_:
        polygon = sample.polygons_[polygonID]
        # polygon.vtk_scalar_ = vtk_scalar
for cellID in df["CellID"].to_numpy():
    # sample.write_cell_collection_vtk(df["CellID"].to_numpy(),"{}.target_cells.vtk".format(iter),use_scalar=True)
    sample.write_cell_collection_vtk([cellID],"{}.single.vtk".format(cellID),use_scalar=False)



0     11
1    157
2    494
3    166
4    378
Name: CellID, dtype: int64


In [31]:
import pandas as pd
import glob
final_iter = 16
dir = "patternA/"
fileA = "{}cellParameters.{}.input".format(dir,final_iter)
# fileB = "{}cellParameters.input".format(dir)
final_iter = 2
dir = "patternB/"
fileB = "{}cellParameters.{}.input".format(dir,final_iter)
def calculate_parameter_space_distance(cellParametersA, cellParametersB):
    df = pd.read_csv(cellParametersA, sep=" ",header=None)
    cellID_to_s0 = {int(i): [float(s0)] for i, s0 in zip(df[0].to_numpy(), df[2].to_numpy())}
    df = pd.read_csv(cellParametersB, sep=" ",header=None)
    for i, row in df.iterrows():
        cellID = row[0]
        s0 = row[2]
        if cellID in cellID_to_s0:
            cellID_to_s0[cellID].append(s0)
        else:
            raise ValueError("CellID {} not found in first pattern".format(cellID))
    distance = 0
    for cellID, s0s in cellID_to_s0.items():
        distance += (s0s[0] - s0s[1])**2
    distance = distance**0.5
    return distance
print("Distance between patterns: {:.010f}".format(calculate_parameter_space_distance(fileA, fileA)))

Distance between patterns: 0.0000000000
